# Edit Product-Family Comparison Plot

This notebook recreates `fig3_product_family_comparison.png` from the local processed data. It is an editable notebook version of `build_product_families()` in the local manuscript helper script:

```text
scripts/analysis/build_manuscript_figures.py
```

The plot compares product-family occurrence at selected energies for DFT/MM, ReaxFF, MACE-medium, and MACE-POLAR-1.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "wall_products.csv").exists():
    ROOT = Path.cwd().parents[1]

DATA = ROOT / "data"
PROCESSED = DATA / "processed"
ASSETS = ROOT / "assets"

dft = pd.read_csv(DATA / "literature" / "dft_energy_resolved_results.csv")
products = pd.read_csv(PROCESSED / "wall_products.csv")
reax = pd.read_csv(PROCESSED / "reaxff_case_summary.csv")

print("DFT rows:", len(dft))
print("MACE product rows:", len(products))
print("ReaxFF case rows:", len(reax))

## Plot Controls

Edit these lists and style values to change the plot.

In [ ]:
SELECTED_ENERGIES = [10, 40, 100]
METHODS = ["DFT-MD", "ReaxFF", "MACE-medium", "MACE-polar"]

DISPLAY = {
    "DFT-MD": "DFT/MM",
    "ReaxFF": "ReaxFF",
    "MACE-medium": "MACE-Medium",
    "MACE-polar": "MACE-Polar",
}

SELECTED_PRODUCTS = [
    "H", "CH3", "F", "HF", "BF2", "BF3", "BF4",
    "C2H2", "C2H3N", "C6H10N2", "C6H11N2",
]

LABEL_MAP = {
    "CH3": r"CH$_3$",
    "BF2": r"BF$_2$",
    "BF3": r"BF$_3$",
    "BF4": r"BF$_4$",
    "C2H2": r"C$_2$H$_2$",
    "C2H3N": r"C$_2$H$_3$N",
    "C6H10N2": r"C$_6$H$_{10}$N$_2$",
    "C6H11N2": r"C$_6$H$_{11}$N$_2$",
}

FIGSIZE = (10.5, 5.8)
DPI = 300
ABSENT_COLOR = "#F2F2F2"
PRESENT_COLOR = "#0072B2"
PRESENT_MARKER = "●"
ABSENT_MARKER = "–"
OUTPUT_STEM = "product_family_comparison_edited"

In [ ]:
def configure_style():
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times", "Nimbus Roman", "DejaVu Serif"],
        "font.size": 13,
        "font.weight": "bold",
        "axes.linewidth": 1.8,
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "xtick.major.size": 6,
        "xtick.major.width": 1.4,
        "xtick.direction": "in",
        "ytick.major.size": 6,
        "ytick.major.width": 1.4,
        "ytick.direction": "in",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.05,
        label,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=15,
        fontweight="bold",
    )

In [ ]:
def present_products(method, energy):
    if method == "DFT-MD":
        value = dft.loc[dft.energy_eV == energy, "reported_products"].iloc[0]
        return set(value.split(";"))
    if method == "ReaxFF":
        values = reax.loc[reax.energy_eV == energy, "final_fragment_formulas"]
        return set(";".join(values).split(";"))
    return set(
        products.loc[
            (products.model == method) & (products.energy_eV == energy),
            "product_formula",
        ]
    )


def build_presence_matrices():
    matrices = {}
    for method in METHODS:
        matrix = np.zeros((len(SELECTED_PRODUCTS), len(SELECTED_ENERGIES)))
        for column, energy in enumerate(SELECTED_ENERGIES):
            present = present_products(method, energy)
            for row, product in enumerate(SELECTED_PRODUCTS):
                matrix[row, column] = product in present
        matrices[method] = matrix
    return matrices


matrices = build_presence_matrices()
matrices.keys()

In [ ]:
def make_product_family_plot(save=True):
    configure_style()
    matrices = build_presence_matrices()
    cmap = ListedColormap([ABSENT_COLOR, PRESENT_COLOR])
    fig, axes = plt.subplots(1, len(METHODS), figsize=FIGSIZE, constrained_layout=True)
    if len(METHODS) == 1:
        axes = [axes]

    for index, (ax, method) in enumerate(zip(axes, METHODS)):
        matrix = matrices[method]
        ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(SELECTED_ENERGIES)), SELECTED_ENERGIES)
        ax.set_xlabel("Impact energy (eV)")
        ax.set_yticks(
            range(len(SELECTED_PRODUCTS)),
            [LABEL_MAP.get(item, item) for item in SELECTED_PRODUCTS] if index == 0 else [],
        )
        ax.set_title(DISPLAY[method], fontweight="bold")
        ax.set_xticks(np.arange(-0.5, len(SELECTED_ENERGIES), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(SELECTED_PRODUCTS), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=1.4)
        ax.tick_params(which="minor", bottom=False, left=False)

        for row in range(len(SELECTED_PRODUCTS)):
            for column in range(len(SELECTED_ENERGIES)):
                observed = bool(matrix[row, column])
                ax.text(
                    column,
                    row,
                    PRESENT_MARKER if observed else ABSENT_MARKER,
                    ha="center",
                    va="center",
                    color="white" if observed else "#777777",
                    fontsize=11,
                    fontweight="bold",
                )
        add_panel_label(ax, chr(ord("A") + index))

    if save:
        ASSETS.mkdir(parents=True, exist_ok=True)
        png = ASSETS / f"{OUTPUT_STEM}.png"
        pdf = ASSETS / f"{OUTPUT_STEM}.pdf"
        fig.savefig(png, dpi=DPI, bbox_inches="tight")
        fig.savefig(pdf, bbox_inches="tight")
        print(f"Saved {png}")
        print(f"Saved {pdf}")
    return fig, axes


fig, axes = make_product_family_plot(save=True)

## Inspect Product Sets

Use these cells to inspect or debug individual method-energy product sets.

In [ ]:
INSPECT_METHOD = "MACE-polar"
INSPECT_ENERGY = 40

sorted(present_products(INSPECT_METHOD, INSPECT_ENERGY))

In [ ]:
presence_rows = []
for method, matrix in build_presence_matrices().items():
    for row, product in enumerate(SELECTED_PRODUCTS):
        for column, energy in enumerate(SELECTED_ENERGIES):
            presence_rows.append({
                "method": method,
                "energy_eV": energy,
                "product": product,
                "observed": bool(matrix[row, column]),
            })

presence_table = pd.DataFrame(presence_rows)
presence_table.pivot_table(index=["product"], columns=["method", "energy_eV"], values="observed", aggfunc="first")